In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact_manual

cwd = Path.cwd()
if (cwd / 'structured_wishart_levy.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'structured_wishart_levy.py').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

import structured_wishart_levy as swl

# Structured Wishart-Levy law

Numerical exploration of `structured_wishart_levy.py`; derivation in
`structured_wishart_levy.md`.

* **Theorem 1** (general two-sided $\tau$): the row field $Y_r(x)$ stays
  functional, density is $\rho(s) = -(1/(\pi s^2))\,\mathrm{Im}\,\langle h_\alpha(Y_r(x,s))\rangle_x$ plus a $1-\gamma$ atom at $0$.
* **Theorem 2** (one-sided $\tau(x,y)=c(y)$): $Y_r(x)$ collapses to a scalar.

The **bottom-left panel** of the widget plots $Y_r(x)$ -- a flat trace
indicates the Theorem 2 hypothesis holds; a bent trace means you are in the
genuine Theorem 1 regime.

Profile families exposed in the widget: `constant`, `one_sided_c`,
`one_sided_r`, `one_sided_two_level`, `one_sided_power_law`,
`separable_sum`, `rank_one_prod`, `block_2x2`, `gaussian_bump`,
`custom_function` (edit `CUSTOM_PROFILE` below). Empirical sampling is CPU
(`scipy.stats.levy_stable`); keep `n_rows` modest in the widget and use
the script for larger sweeps.

In [ ]:
DEFAULT_PARAMS = dict(
    alpha=1.5,
    gamma=0.7,
    profile='separable_sum',
    constant_value=1.0,
    a_x=0.5,
    a_y=0.5,
    center_x=0.5,
    center_y=0.5,
    width=0.25,
    block_tl=0.7,
    block_tr=1.3,
    block_bl=1.3,
    block_br=0.7,
    c_left=0.75,
    c_right=1.5,
    split=0.5,
    exponent=0.5,
    cutoff=0.05,
    entry_scale=1.0,
    normalization='stable',
    n_rows=200,
    num_matrices=8,
    bins=81,
    s_max=6.0,
    num_points=81,
    imag_eps=1e-2,
    quadrature_order=64,
    profile_order=24,
    tail_start=2.0,
    seed=0,
)

DEFAULT_PARAMS

In [ ]:
def CUSTOM_PROFILE(X, Y):
    """Edit this function and then choose profile='custom_function' below.
    X, Y arrive as 2-D meshgrids of shape (n_rows, n_cols)."""
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    return 1.0 + 0.4 * np.sin(2 * np.pi * X) * np.cos(2 * np.pi * Y)


PROFILE_FAMILIES = [
    'constant',
    'one_sided_c', 'one_sided_r',
    'one_sided_two_level', 'one_sided_power_law',
    'separable_sum', 'rank_one_prod',
    'block_2x2', 'gaussian_bump',
    'custom_function',
]


def build_profile(name, p):
    """Return a callable tau(X, Y) for the named family using params dict p."""
    c = p['constant_value']
    if name == 'constant':
        return lambda X, Y: np.full(np.broadcast_shapes(X.shape, Y.shape), c)
    if name == 'one_sided_c':
        return lambda X, Y: c + p['a_y'] * Y + 0 * X
    if name == 'one_sided_r':
        return lambda X, Y: c + p['a_x'] * X + 0 * Y
    if name == 'one_sided_two_level':
        return lambda X, Y: np.where(Y < p['split'], p['c_left'], p['c_right']) + 0 * X
    if name == 'one_sided_power_law':
        return lambda X, Y: c * np.maximum(Y, p['cutoff']) ** (-p['exponent']) + 0 * X
    if name == 'separable_sum':
        return lambda X, Y: c + p['a_x'] * X + p['a_y'] * Y
    if name == 'rank_one_prod':
        return lambda X, Y: (c + p['a_x'] * X) * (c + p['a_y'] * Y)
    if name == 'block_2x2':
        def tau(X, Y):
            T = np.empty(np.broadcast_shapes(X.shape, Y.shape), dtype=float)
            X_, Y_ = np.broadcast_arrays(X, Y)
            top, left = Y_ >= 0.5, X_ < 0.5
            T[ top &  left] = p['block_tl']
            T[ top & ~left] = p['block_tr']
            T[~top &  left] = p['block_bl']
            T[~top & ~left] = p['block_br']
            return T
        return tau
    if name == 'gaussian_bump':
        return lambda X, Y: c + np.exp(
            -((X - p['center_x'])**2 + (Y - p['center_y'])**2)
            / (2.0 * p['width']**2)
        )
    if name == 'custom_function':
        return CUSTOM_PROFILE
    raise ValueError(f'unknown profile family: {name!r}')


PROFILE_FAMILIES

In [ ]:
# Validation gates (smoke test): run these before exploring the widget.
# Both gates use the shared-rule g/h, so agreement should be ~1e-9 at
# both field and density level.

print('Gate 1  (Corollary 1: tau == 1  ->  wishart_levy)')
g1 = swl.compare_constant_to_wishart(
    alpha=1.5, gamma=0.6, s_max=4.0, num_points=41,
    imag_eps=1e-2, quadrature_order=64, profile_order=16,
)
for k, v in g1.items():
    print(f'    {k:38s} {v:.2e}')

print('\nGate 2  (Theorem 2: one-sided tau  ->  scalar Y_r closure)')
g2 = swl.compare_one_sided_to_scalar_closure(
    alpha=1.4, gamma=0.5,
    c_profile=lambda y: 0.7 + 0.6 * y,
    s_max=4.0, num_points=41, imag_eps=1e-2,
    quadrature_order=64, profile_order=32,
)
for k, v in g2.items():
    print(f'    {k:38s} {v:.2e}')

In [ ]:
def plot_structured_comparison(
    alpha, gamma, profile,
    constant_value, a_x, a_y, center_x, center_y, width,
    block_tl, block_tr, block_bl, block_br,
    c_left, c_right, split, exponent, cutoff,
    entry_scale, normalization,
    n_rows, num_matrices, bins,
    s_max, num_points, imag_eps, quadrature_order, profile_order,
    tail_start, seed,
    show_empirical=True, show_theory=True, show_tail=True,
):
    params = dict(
        constant_value=float(constant_value),
        a_x=float(a_x), a_y=float(a_y),
        center_x=float(center_x), center_y=float(center_y), width=float(width),
        block_tl=float(block_tl), block_tr=float(block_tr),
        block_bl=float(block_bl), block_br=float(block_br),
        c_left=float(c_left), c_right=float(c_right),
        split=float(split), exponent=float(exponent), cutoff=float(cutoff),
    )
    tau = build_profile(str(profile), params)

    theory = (
        swl.theoretical_structured_singular_value_curve(
            float(alpha), float(gamma), tau,
            s_max=float(s_max), num_points=int(num_points),
            entry_scale=float(entry_scale), normalization=str(normalization),
            imag_eps=float(imag_eps),
            quadrature_order=int(quadrature_order),
            profile_order=int(profile_order),
        ) if show_theory else None
    )

    empirical = (
        swl.empirical_structured_singular_value_spectrum(
            float(alpha), float(gamma), tau,
            n_rows=int(n_rows), num_matrices=int(num_matrices),
            entry_scale=float(entry_scale), normalization=str(normalization),
            bins=int(bins), seed=int(seed),
            singular_range=(0.0, float(s_max)),
        ) if show_empirical else None
    )

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    ax_tau, ax_dens, ax_yrow, ax_check = axes.flat

    # top-left: tau heatmap
    n_viz = 64
    xv = (np.arange(n_viz) + 0.5) / n_viz
    Xv, Yv = np.meshgrid(xv, xv, indexing='ij')
    Tviz = np.asarray(tau(Xv, Yv), dtype=float)
    im = ax_tau.imshow(
        Tviz, origin='lower', extent=[0, 1, 0, 1],
        aspect='auto', cmap='viridis',
    )
    plt.colorbar(im, ax=ax_tau, fraction=0.046, pad=0.04)
    title_tau = rf'$\tau(x,y)$  [{profile}]'
    if theory is not None:
        title_tau += rf';  $\iint|\tau|^\alpha\,dx\,dy = {theory.profile_alpha_moment:.3f}$'
    ax_tau.set_title(title_tau)
    ax_tau.set_xlabel('y (col)')
    ax_tau.set_ylabel('x (row)')

    # top-right: density (log y)
    if empirical is not None:
        ax_dens.bar(
            empirical.sv_bin_centers, empirical.sv_density,
            width=np.diff(empirical.sv_bin_edges),
            alpha=0.35, color='C0',
            label=f'empirical (N={empirical.n_rows}, reps={empirical.num_matrices})',
        )
    if theory is not None:
        ax_dens.plot(
            theory.singular_values, theory.singular_density,
            'C3-', lw=2, label='theory (Thm 1 collapse)',
        )
    if show_tail:
        s_tail = np.linspace(max(float(tail_start), 1e-3), float(s_max), 400)
        tail = swl.asymptotic_singular_density(
            s_tail, float(alpha), float(gamma), tau,
            entry_scale=float(entry_scale),
            normalization=str(normalization),
            profile_order=int(profile_order),
        )
        ax_dens.plot(s_tail, tail, 'k--', lw=1, label=r'tail $\sim s^{-1-\alpha}$')
    ax_dens.set_yscale('log')
    if empirical is not None:
        pos = empirical.sv_density[empirical.sv_density > 0]
        if pos.size:
            ax_dens.set_ylim(pos.min() * 0.5, pos.max() * 2)
    ax_dens.set_xlabel('s')
    ax_dens.set_ylabel(r'$\rho(s)$')
    ax_dens.set_title(rf'singular value density   ($\alpha={alpha:.2f}$, $\gamma={gamma:.2f}$)')
    ax_dens.legend(loc='best', fontsize=9)

    # bottom-left: Y_r(x) at the largest finite s
    if theory is not None:
        finite = np.where(np.all(np.isfinite(theory.y_row), axis=1))[0]
        if finite.size:
            idx = int(finite.max())
            s_pick = theory.singular_values[idx]
            yr = theory.y_row[idx]
            spread = float(np.ptp(np.abs(yr)))
            ax_yrow.plot(theory.row_nodes, yr.real, 'C0-o', ms=3, label=r'$\mathrm{Re}\,Y_r$')
            ax_yrow.plot(theory.row_nodes, yr.imag, 'C3-s', ms=3, label=r'$\mathrm{Im}\,Y_r$')
            ax_yrow.axhline(yr.real.mean(), color='C0', ls=':', alpha=0.5)
            ax_yrow.axhline(yr.imag.mean(), color='C3', ls=':', alpha=0.5)
            verdict = '(flat -> Thm 2 holds)' if spread < 1e-10 else '(bent -> genuine Thm 1)'
            ax_yrow.set_title(
                rf'$Y_r(x)$ at $s={s_pick:.2f}$;  $\sup|Y_r|-\inf|Y_r|={spread:.2e}$  {verdict}'
            )
        else:
            ax_yrow.text(0.5, 0.5, 'no finite $Y_r$ in theory.y_row',
                         transform=ax_yrow.transAxes, ha='center')
    else:
        ax_yrow.text(0.5, 0.5, 'enable "Theory curve" to see $Y_r(x)$',
                     transform=ax_yrow.transAxes, ha='center')
    ax_yrow.set_xlabel('x (row coord)')
    ax_yrow.set_ylabel(r'$Y_r(x)$')
    ax_yrow.legend(loc='best', fontsize=9)

    # bottom-right: mass & tail consistency
    if theory is not None:
        s = theory.singular_values
        d = np.nan_to_num(theory.singular_density)
        if s.size > 1:
            mids = 0.5 * (d[1:] + d[:-1])
            cum = np.concatenate([[0.0], np.cumsum(mids * np.diff(s))])
        else:
            cum = np.zeros_like(s)
        ax_check.plot(s, cum, 'C2-', label=r"$\int_0^s\rho\,ds'$")
        ax_check.axhline(theory.gamma, color='k', ls='--', lw=0.8,
                         label=rf'$\gamma={theory.gamma:.2f}$ (Gram mass)')
        ax_check.set_ylim(0, max(theory.gamma * 1.1, float(cum.max()) * 1.1))
        ax2 = ax_check.twinx()
        with np.errstate(invalid='ignore'):
            tail_ratio = d * s ** (1.0 + float(alpha))
        ax2.plot(s, tail_ratio, 'C4-', label=r'$s^{1+\alpha}\rho(s)$')
        B = swl.structured_singular_tail_prefactor_from_curve(theory)
        ax2.axhline(B, color='C4', ls=':', lw=0.8, label=f'B = {B:.3f}')
        ax2.set_ylabel(r'$s^{1+\alpha}\rho(s)$', color='C4')
        ax2.tick_params(axis='y', colors='C4')
        ax2.legend(loc='center right', fontsize=8)
    ax_check.set_xlabel('s')
    ax_check.set_ylabel('cumulative mass', color='C2')
    ax_check.tick_params(axis='y', colors='C2')
    ax_check.legend(loc='lower right', fontsize=8)
    ax_check.set_title('mass & tail consistency')

    plt.tight_layout()
    plt.show()


@interact_manual(
    alpha=widgets.FloatText(value=DEFAULT_PARAMS['alpha']),
    gamma=widgets.FloatText(value=DEFAULT_PARAMS['gamma']),
    profile=widgets.Dropdown(options=PROFILE_FAMILIES, value=DEFAULT_PARAMS['profile']),
    constant_value=widgets.FloatText(value=DEFAULT_PARAMS['constant_value']),
    a_x=widgets.FloatText(value=DEFAULT_PARAMS['a_x']),
    a_y=widgets.FloatText(value=DEFAULT_PARAMS['a_y']),
    center_x=widgets.FloatText(value=DEFAULT_PARAMS['center_x']),
    center_y=widgets.FloatText(value=DEFAULT_PARAMS['center_y']),
    width=widgets.FloatText(value=DEFAULT_PARAMS['width']),
    block_tl=widgets.FloatText(value=DEFAULT_PARAMS['block_tl']),
    block_tr=widgets.FloatText(value=DEFAULT_PARAMS['block_tr']),
    block_bl=widgets.FloatText(value=DEFAULT_PARAMS['block_bl']),
    block_br=widgets.FloatText(value=DEFAULT_PARAMS['block_br']),
    c_left=widgets.FloatText(value=DEFAULT_PARAMS['c_left']),
    c_right=widgets.FloatText(value=DEFAULT_PARAMS['c_right']),
    split=widgets.FloatText(value=DEFAULT_PARAMS['split']),
    exponent=widgets.FloatText(value=DEFAULT_PARAMS['exponent']),
    cutoff=widgets.FloatText(value=DEFAULT_PARAMS['cutoff']),
    entry_scale=widgets.FloatText(value=DEFAULT_PARAMS['entry_scale']),
    normalization=widgets.Dropdown(options=['stable', 'belinschi'],
                                   value=DEFAULT_PARAMS['normalization']),
    n_rows=widgets.IntText(value=DEFAULT_PARAMS['n_rows']),
    num_matrices=widgets.IntText(value=DEFAULT_PARAMS['num_matrices']),
    bins=widgets.IntText(value=DEFAULT_PARAMS['bins']),
    s_max=widgets.FloatText(value=DEFAULT_PARAMS['s_max']),
    num_points=widgets.IntText(value=DEFAULT_PARAMS['num_points']),
    imag_eps=widgets.FloatText(value=DEFAULT_PARAMS['imag_eps']),
    quadrature_order=widgets.IntText(value=DEFAULT_PARAMS['quadrature_order']),
    profile_order=widgets.IntText(value=DEFAULT_PARAMS['profile_order']),
    tail_start=widgets.FloatText(value=DEFAULT_PARAMS['tail_start']),
    seed=widgets.IntText(value=DEFAULT_PARAMS['seed']),
    show_empirical=widgets.Checkbox(value=True, description='Empirical SVD'),
    show_theory=widgets.Checkbox(value=True, description='Theory curve'),
    show_tail=widgets.Checkbox(value=True, description='Tail asymptote'),
)
def explore_structured_singular_values(**kw):
    plot_structured_comparison(**kw)